In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
train=pd.read_csv('/content/train.csv')
test=pd.read_csv('/content/test.csv')

In [3]:
train.head()

,id,class,cap-diameter,cap-shape,cap-surface,cap-color,does-bruise-or-bleed,gill-attachment,gill-spacing,gill-color,...,stem-root,stem-surface,stem-color,veil-type,veil-color,has-ring,ring-type,spore-print-color,habitat,season
0,0,e,8.80,f,s,u,f,a,c,w,...,NaN,NaN,w,NaN,NaN,f,f,NaN,d,a
1,1,p,4.51,x,h,o,f,a,c,n,...,NaN,y,o,NaN,NaN,t,z,NaN,d,w
2,2,e,6.94,f,s,b,f,x,c,w,...,NaN,s,n,NaN,NaN,f,f,NaN,l,w
3,3,e,3.88,f,y,g,f,s,NaN,g,...,NaN,NaN,w,NaN,NaN,f,f,NaN,d,u
4,4,e,5.85,x,l,w,f,d,NaN,w,...,NaN,NaN,w,NaN,NaN,f,f,NaN,g,a


In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3116945 entries, 0 to 3116944
Data columns (total 22 columns):
 #   Column                Dtype  
---  ------                -----  
 0   id                    int64  
 1   class                 object 
 2   cap-diameter          float64
 3   cap-shape             object 
 4   cap-surface           object 
 5   cap-color             object 
 6   does-bruise-or-bleed  object 
 7   gill-attachment       object 
 8   gill-spacing          object 
 9   gill-color            object 
 10  stem-height           float64
 11  stem-width            float64
 12  stem-root             object 
 13  stem-surface          object 
 14  stem-color            object 
 15  veil-type             object 
 16  veil-color            object 
 17  has-ring              object 
 18  ring-type             object 
 19  spore-print-color     object 
 20  habitat               object 
 21  season                object 
dtypes: float64(3), int64(1), object(18)
memory

In [5]:
train.isnull().sum().sort_values(ascending=False).head(20)

,0
veil-type,2957493
spore-print-color,2849682
stem-root,2757023
veil-color,2740947
stem-surface,1980861
gill-spacing,1258435
cap-surface,671023
gill-attachment,523936
ring-type,128880
gill-color,57


In [6]:
X=train.drop(columns=['class'])
y=train['class']

In [7]:
X.dtypes.value_counts()

,count
object,17
float64,3
int64,1


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [9]:
encoder=OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)
X_encoded=encoder.fit_transform(X)

In [10]:
X_train,X_valid,y_train,y_valid=train_test_split(X_encoded,y,test_size=0.2,random_state=42)

In [ ]:
model=RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train,y_train)

In [ ]:
preds = model.predict(X_valid)

print(
    "Accuracy:",
    accuracy_score(y_valid, preds)
)

In [ ]:
importance = pd.DataFrame({
    "feature":X.columns,
    "importance":model.feature_importances_
})

In [ ]:
importance.sort_values(by="importance",ascending=False)

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    model,
    X_encoded,
    y,
    cv=5,
    scoring="accuracy"
)

print(scores)
print(scores.mean())

### Generate Submission File

Finally, we will encode the test data using the trained `OrdinalEncoder`, make predictions with the `RandomForestClassifier` model, and save the results to a `submission.csv` file. This file will contain the `id` from the test set and the predicted `class`.

In [ ]:
# Encode the test data using the fitted encoder
test_encoded = encoder.transform(test)

# Make predictions on the encoded test data
predictions = model.predict(test_encoded)

# Create a DataFrame for the submission file
submission = pd.DataFrame({
    "id": test["id"],
    "class": predictions
})

# Display the shape and first few rows of the submission DataFrame
print("Submission DataFrame shape:", submission.shape)
print("Submission DataFrame head:\n", submission.head())

# Save the submission DataFrame to a CSV file
submission.to_csv("submission.csv", index=False)
print("Submission file 'submission.csv' generated successfully.")